In [214]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.optimize import curve_fit
import scipy.constants as pc
import astropy.units as u
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy.table import Table
from astropy.table import vstack
from astropy.cosmology import FlatLambdaCDM
from scipy.stats import ttest_rel
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.ticker import AutoMinorLocator
import os
from matplotlib.gridspec import GridSpec

FIG_DIR = "/raid/scratch/work/alberttg/Masters_project/Plots"

DATA_PRODUCTS_PATH = "/raid/scratch/work/alberttg/Masters_project/Data_products"

In [215]:
def fits_table(file_path):
        # Open the FITS file
        with fits.open(file_path) as hdul:
            # Show the HDU structure
            # hdul.info()
            # Usually the table is in extension 1
            data = hdul[1].data

        # Convert to an Astropy Table
        tbl = Table(data)
        #print("Number of objects = ", len(tbl))

        # Print column names
        # print("\nColumns:")
        # print(tbl.colnames)

        return tbl

In [216]:
def add_marginal_hists(ax, x, y, bins=40, size="15%", pad=0.1, color="gray"):
    """
    Add marginal histograms of x and y on top of and to the right of ax,
    without disturbing ax's existing content (scatter, legend, etc.).

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axes to attach histograms to.
    x, y : array-like
        Data for the x and y histograms.
    bins : int
        Number of histogram bins.
    size : str
        Size of the marginal axes relative to ax (e.g. "15%").
    pad : float
        Padding between ax and the marginal axes.
    color : str
        Histogram color.

    Returns
    -------
    ax_histx, ax_histy : the two new marginal axes
    """
    divider = make_axes_locatable(ax)

    # Top histogram (x-axis distribution)
    ax_histx = divider.append_axes("top", size=size, pad=pad, sharex=ax)
    ax_histx.hist(x, bins=bins, color=color, edgecolor='0.2', linewidth=0.5)
    ax_histx.tick_params(axis="x", labelbottom=False)  # hide x labels (shared with ax)
    ax_histx.set_yticks([])
    for spine in ["top", "right", "left"]:
        ax_histx.spines[spine].set_visible(False)

    # Right histogram (y-axis distribution)
    ax_histy = divider.append_axes("right", size=size, pad=pad, sharey=ax)
    ax_histy.hist(y, bins=bins, orientation="horizontal", color=color, edgecolor='0.2', linewidth=0.5)
    ax_histy.tick_params(axis="y", labelleft=False)  # hide y labels (shared with ax)
    ax_histy.set_xticks([])
    for spine in ["top", "right", "bottom"]:
        ax_histy.spines[spine].set_visible(False)

    return ax_histx, ax_histy

In [217]:
def plot_binned_medians_for_distribution(
    ax, x, y,
    x_upper_err, x_lower_err, y_upper_err, y_lower_err,
    n_bins, bin_axis, bin_axis_name,
    marker="s", markersize=14, markeredgecolor="black",
    markeredgewidth=1.5, color="red", **errorbar_kwargs
):
    """
    Bin data along `bin_axis`, compute the median of x and y within each bin,
    and plot the medians on `ax` with error bars derived from per-point
    error arrays. Marker style: thick-edged star. Not defined bin widths,
    instead displays the distribution of points. Is a bit dodgy.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axes to plot on (existing content is preserved).
    x, y : array-like
        Data points.
    x_upper_err, x_lower_err, y_upper_err, y_lower_err : array-like
        Per-point asymmetric errors, same length as x/y.
    n_bins : int or array-like
        Desired number of bins, or explicit bin edges. If an int, it will
        be reduced automatically (see `min_points_per_bin`) and bin edges
        are chosen so each bin contains roughly equal numbers of points.
        If array-like, used as explicit bin edges (no automatic adjustment).
    bin_axis : {"x", "y"}
        Which variable to bin on.
    bin_axis_name:
        For the label on the median of binned points.
    marker, markersize, markeredgecolor, markeredgewidth, color :
        Passed to ax.errorbar for marker styling.
    **errorbar_kwargs :
        Any additional kwargs forwarded to ax.errorbar.

    Returns
    -------
    bin_x, bin_y, xerr, yerr : ndarrays
        The computed bin medians and their [lower, upper] error arrays
        (xerr and yerr have shape (2, n_bins)).
    """
    x = np.asarray(x)
    y = np.asarray(y)

    x_upper_err = np.zeros_like(x) if x_upper_err is None else np.asarray(x_upper_err)
    x_lower_err = np.zeros_like(x) if x_lower_err is None else np.asarray(x_lower_err)

    y_upper_err = np.zeros_like(y) if y_upper_err is None else np.asarray(y_upper_err)
    y_lower_err = np.zeros_like(y) if y_lower_err is None else np.asarray(y_lower_err)

    bin_data = x if bin_axis == "x" else y

    if np.isscalar(n_bins):
        n_data = len(bin_data)
        # Equal-count (quantile) bin edges, so each bin gets ~n_data/n_bins points
        sorted_data = np.sort(bin_data)
        quantile_positions = np.linspace(0, n_data - 1, n_bins + 1)
        edges = np.interp(quantile_positions, np.arange(n_data), sorted_data)
        # Guard against duplicate edges (e.g. many repeated values)
        edges = np.unique(edges)
        n_bins = len(edges) - 1
    else:
        edges = np.asarray(n_bins)
        n_bins = len(edges) - 1

    bin_idx = np.digitize(bin_data, edges) - 1
    bin_idx = np.clip(bin_idx, 0, n_bins - 1)  # ensure max value falls in last bin


    def bootstrap_median_errors(values, n_boot=5000, random_state=None):
        """
        Bootstrap estimate of asymmetric uncertainty on the median.

        Returns
        -------
        median, upper_err, lower_err
        """
        values = np.asarray(values)
        rng = np.random.default_rng(random_state)

        n = len(values)

        boot = np.empty(n_boot)

        for i in range(n_boot):
            sample = rng.choice(values, size=n, replace=True)
            boot[i] = np.median(sample)

        median = np.median(values)

        lo, hi = np.percentile(boot, [16, 84])

        return median, hi - median, median - lo


    bin_x, bin_y = [], []
    xerr_lo, xerr_hi = [], []
    yerr_lo, yerr_hi = [], []

    for i in range(n_bins):
        mask = bin_idx == i
        n = mask.sum()

        x_med, x_hi, x_lo = bootstrap_median_errors(x[mask])
        y_med, y_hi, y_lo = bootstrap_median_errors(y[mask])

        bin_x.append(x_med)
        xerr_lo.append(x_lo)
        xerr_hi.append(x_hi)

        bin_y.append(y_med)
        yerr_lo.append(y_lo)
        yerr_hi.append(y_hi)
    

    bin_x = np.array(bin_x)
    bin_y = np.array(bin_y)
    xerr = np.array([xerr_lo, xerr_hi])
    yerr = np.array([yerr_lo, yerr_hi])

    ax.errorbar(
        bin_x, bin_y, xerr=xerr, yerr=yerr,
        fmt=marker, markersize=markersize,
        markeredgecolor=markeredgecolor, markeredgewidth=markeredgewidth,
        color=color, linestyle="none", **errorbar_kwargs,
        label=f"Median of binned {bin_axis_name}", zorder=3
    )

    return bin_x, bin_y, xerr, yerr


In [218]:
def redshift_bins(redshifts, y_values, ax, marker_style, marker_size, colour, edgecolor='black', edgewidth=1):
    """
    Bin the data by redshift and compute the median and bootstrap errors for each bin.

    Parameters
    ----------
    redshifts : array-like
        The redshift values.
    y_values : array-like
        The corresponding y-values to compute statistics on.
    ax : matplotlib.axes.Axes
        The axes on which to plot the binned data.
    marker_style : str
        Square or diamond or circle points for the plot.
    marker_size : float
        Size of the points for the plot.
    colour : str
        Color of the points for the plot.
    """
    bins = [[6.5, 7.5], [7.5, 8.5], [8.5, 9.5], [9.5, 10.5], [10.5, 11.5], [11.5, 13.5]]
    # We've set z=6.5 as the minimum and the highest well-measured redshift galaxies at 13.5.

    bin_centers = []
    medians = []
    upper_errors = []
    lower_errors = []

    for i in range(len(bins)):
        bin_mask = (redshifts >= bins[i][0]) & (redshifts < bins[i][1])
        bin_redshifts = redshifts[bin_mask]
        bin_y_values = y_values[bin_mask]

        if len(bin_y_values) > 5:
            # Stops error message when there are no galaxies in a bin
            median = np.median(bin_y_values)
            upper_err = np.percentile(bin_y_values, 84) - median
            lower_err = median - np.percentile(bin_y_values, 16)
            bin_center = np.mean(bins[i])

            bin_centers.append(bin_center)
            medians.append(median)
            upper_errors.append(upper_err)
            lower_errors.append(lower_err)

    ax.errorbar(bin_centers, medians, yerr=[lower_errors, upper_errors], 
                fmt=marker_style, markersize=marker_size,
                markeredgecolor=edgecolor, markeredgewidth=edgewidth,
                color=colour, linestyle="none",
                label=f"Median of binned redshifts", zorder=3
            )
    
    return bin_centers, medians, upper_errors, lower_errors


In [219]:
def plot_median_bins(x_values, y_values, n_bins, binned_axis_name, ax, marker_style, marker_size, colour):
    """
    Bin the data by redshift and compute the median and 16th and 84th percentiles for each bin.
    Ensures there must be at least 5 galaxies in a bin.
    Plots the median values with error bars on the provided axes.

    Parameters
    ----------
    x_values : array-like
        The x-values to bin.
    y_values : array-like
        The corresponding y-values to compute statistics on.
    n_bins : int
        The number of bins to use for binning the data.
    ax : matplotlib.axes.Axes
        The axes on which to plot the binned data.
    marker_style : str
        Square or diamond or circle points for the plot.
    marker_size : float
        Size of the points for the plot.
    colour : str
        Color of the points for the plot.
    """
    bins = np.linspace(x_values.min(), x_values.max(), n_bins + 1)

    bins = [[bins[i], bins[i + 1]] for i in range(len(bins) - 1)]

    bin_centers = []
    medians = []
    upper_errors = []
    lower_errors = []

    for i in range(len(bins)):
        bin_mask = (x_values >= bins[i][0]) & (x_values < bins[i][1])
        bin_x_values = x_values[bin_mask]
        bin_y_values = y_values[bin_mask]

        if len(bin_y_values) > 5:
            # Stops error message when there are no galaxies in a bin
            median = np.median(bin_y_values)
            upper_err = np.percentile(bin_y_values, 84) - median
            lower_err = median - np.percentile(bin_y_values, 16)
            bin_center = np.mean(bins[i])

            bin_centers.append(bin_center)
            medians.append(median)
            upper_errors.append(upper_err)
            lower_errors.append(lower_err)

    ax.errorbar(bin_centers, medians, yerr=[lower_errors, upper_errors], 
                fmt=marker_style, markersize=marker_size,
                markeredgecolor='black', markeredgewidth=1,
                color=colour, linestyle="none",
                label=f"Median of binned {binned_axis_name}", zorder=3
            )
    
    return bin_centers, medians, upper_errors, lower_errors


In [220]:
def open_catalogue_data(path):
  """
  Reads in a catalogue of data from a FITS file and returns the data as an Astropy Table.
  """

  hdul = fits.open(path)
  hdu_names = [hdu.name for hdu in hdul]

  # sex_cat = [hdu for hdu, name in zip(hdul, hdu_names) if name == "OBJECTS"][0]
    
  # EPOCHS series (Conselice+24, Adams+24, Austin+25, Harvey+25)

  sex_tab = Table.read(path, hdu = "OBJECTS")
  sex_tab_colnames = sex_tab.colnames
  # print(sex_tab_colnames)
  # sky position (Ra = ALPHA_J2000, Dec = DELTA_J2000)
  # flux columns: FLUX_APER_{band}_aper_corr_Jy [Jy]
  # flux error columns: FLUXERR_APER_{band}_loc_depth_5pc_Jy [Jy]

  bagpipes_zfix = Table.read(path, hdu = "BAGPIPES_SFH_CONT_BURSTY_ZEAZYSFHZBLUEAGN_3.0,10.0MYR_CALZETTI_LOG_10_Z_LOG_10_BPASS_ZFIX")
  # ext 3

  bagpipes_zgauss = Table.read(path, hdu = "BAGPIPES_SFH_CONT_BURSTY_ZEAZYSFHZBLUEAGN_3.0,10.0MYR_CALZETTI_LOG_10_Z_LOG_10_BPASS_ZGAUSS_3.0SIG")
  # ext 4

  selection_tab = Table.read(path, hdu = "SELECTION")

  return sex_tab, bagpipes_zfix, bagpipes_zgauss, selection_tab

In [221]:
def masked_table(sex_tab, bagpipes_zfix, bagpipes_zgauss, selection_tab):
    """
    Filtered zgauss table for galaxies with z >=6.5.
    Found the corresponding galaxy ids in zfix.
    Added the zgauss redshift error values to the zfix table.
    Highlight LRDs.
    Remove brown dwarfs.
    """
    # --------------- Selecting high redshift galaxies ---------------
    survey_ids = bagpipes_zgauss["#ID"]
    # The ids of galaxies in zgauss table with redshift >= 6.5, which are the same in zfix

    zfix_match = bagpipes_zfix[np.isin(bagpipes_zfix["#ID"], survey_ids)]
    # Matches z fix galaxies to ones in z gauss
    z_spec_mask = zfix_match["input_redshift"] >= 6.5

    zfix_sample = zfix_match[z_spec_mask]
    # Now our sample is filtered for redshift >= 6.5

    print("Check inital length: ", len(zfix_sample))

    # ----------- select brown dwarfs ----------
    brown_dwarf_mask = selection_tab["dchi2_EAZY_sfhz_blue_agn>0.0_F444W_bd_0.32as"] # These are brown dwarfs
    selection_tab = selection_tab[~brown_dwarf_mask] # We dont want BDs

    selection_tab_ids = selection_tab["NUMBER"].data

    zfix_sample = zfix_sample[np.isin(zfix_sample["#ID"].data, selection_tab_ids)]

    print("Check length after filtering out brown dwarfs: ", len(zfix_sample))


    zfix_sample["photo_z"] = np.zeros(len(zfix_sample), dtype=float)
    zfix_sample["photo_z_lower_err"] = np.zeros(len(zfix_sample), dtype=float)
    zfix_sample["photo_z_upper_err"] = np.zeros(len(zfix_sample), dtype=float)
    zfix_sample["is_LRD"] = np.zeros(len(zfix_sample), dtype=bool)
    # We want a column for photo_z in the zfix table to store the zgauss redshift values.
    # And a column for is_LRD to store the selection results.

    for i in range(len(zfix_sample)):

        id = zfix_sample["#ID"][i]

        # --------------- Data from zgauss into zfix table ---------------
        if id in bagpipes_zgauss["#ID"].data:

            z_50 = bagpipes_zgauss["redshift_50"][np.where(id == bagpipes_zgauss["#ID"].data)[0][0]]
            # Getting zgauss photo z into z fix table
            z_84 = bagpipes_zgauss["redshift_84"][np.where(id == bagpipes_zgauss["#ID"].data)[0][0]]
            z_16 = bagpipes_zgauss["redshift_16"][np.where(id == bagpipes_zgauss["#ID"].data)[0][0]]
            # Using zgauss errors but plotting zfix redshift values.
            zfix_sample["photo_z_lower_err"][i] = z_50 - z_16
            zfix_sample["photo_z_upper_err"][i] = z_84 - z_50

        # --------------- Data from selection into zfix table -------------
        if id in selection_tab["NUMBER"].data:

            # ------------ highlight LRDs -------------
            is_LRD =  selection_tab["Kokorev+24_LRD_red2_0.32as"][np.where(id == selection_tab["NUMBER"].data)[0][0]] & \
                        selection_tab["compact_F444W_r=0.32/0.16as<1.10*psf"][np.where(id == selection_tab["NUMBER"].data)[0][0]]

            zfix_sample["is_LRD"][i] = is_LRD

    sample_table_path = os.path.join(DATA_PRODUCTS_PATH, "First_sample_table.fits")
    zfix_sample.write(sample_table_path, format="fits", overwrite=True)

    return zfix_sample


In [222]:
def plot_stellar_mass_vs_redshift(sample_table):
    """
    Shows survey by colour and shape.
    """

    log_stellar_mass_Q16 = sample_table["stellar_mass_16"]
    log_stellar_mass_Q50 = sample_table["stellar_mass_50"]
    log_stellar_mass_Q84 = sample_table["stellar_mass_84"]

    log_stellar_mass_upper_err = log_stellar_mass_Q84 - log_stellar_mass_Q50
    log_stellar_mass_lower_err = log_stellar_mass_Q50 - log_stellar_mass_Q16

    spec_z = sample_table["input_redshift"]

    fig, ax = plt.subplots(figsize=(10, 10))

    # --------------------------- Colouring and marker styles for each survey ----------------------------
    survey = np.asarray(sample_table["SURVEY"].data, dtype=str)
    south = np.asarray(["JADES-DR3-GS-East", "JADES-DR3-GS-North", "JADES-DR3-GS-South", "JADES-DR3-GS-West", "NGDEEP"], dtype=str)
    north = np.asarray(["JADES-DR3-GN-Parallel", "JADES-DR3-GN-Medium", "JADES-DR3-GN-Deep"], dtype=str)
    uds =  np.asarray(["PRIMER-UDS"], dtype=str)
    cosmos = np.asarray(["PRIMER-COSMOS"], dtype=str)
    ceers = np.asarray(["CEERSP1", "CEERSP2", "CEERSP3", "CEERSP4", "CEERSP5", "CEERSP6", "CEERSP7", "CEERSP8", "CEERSP9", "CEERSP10"], dtype=str)
    nep = np.asarray(["NEP-1", "NEP-2", "NEP-3", "NEP-4"], dtype=str)

    colours = np.empty(len(survey), dtype=object)
    marker_styles = np.empty(len(survey), dtype='<U1')  # Use a string type for marker styles
    for i, s in enumerate(survey):
        if s in south:
            colours[i] = 'tab:orange'
            marker_styles[i] = 'o'
        if s in north:
            colours[i] = 'tab:cyan'
            marker_styles[i] = 'd'
        if s in ceers:
            colours[i] = 'tab:purple'
            marker_styles[i] = 'p'
        if s in uds:
            colours[i] = 'tab:blue'
            marker_styles[i] = '^'
        if s in cosmos:
            colours[i] = 'tab:red'
            marker_styles[i] = '*'
        if s in nep:
            colours[i] = 'tab:green'
            marker_styles[i] = 's'

    legend_elements = [
            Line2D([0], [0], marker='o', color='w',
                label='GOODS-South',
                markerfacecolor='tab:orange',
                markeredgecolor='0.3',
                linewidth=0.5,
                markersize=7),

            Line2D([0], [0], marker='o', color='w',
                label='GOODS-North',
                markerfacecolor='tab:cyan',
                markeredgecolor='0.3',
                linewidth=0.5,
                markersize=7),
                
            Line2D([0], [0], marker='p', color='w',
                label='CEERS',
                markerfacecolor='tab:purple',
                markeredgecolor='0.3',
                linewidth=0.5,
                markersize=7),
                    
            Line2D([0], [0], marker='^', color='w',
                label='PRIMER-UDS',
                markerfacecolor='tab:blue',
                markeredgecolor='0.3',
                linewidth=0.5,
                markersize=7),
                    
            Line2D([0], [0], marker='s', color='w',
                label='NEP',
                markerfacecolor='tab:green',
                markeredgecolor='0.3',
                linewidth=0.5,
                markersize=7),
                    
            Line2D([0], [0], marker='*', color='w',
                label='PRIMER-COSMOS',
                markerfacecolor='tab:red',
                markeredgecolor='0.3',
                linewidth=0.5,
                markersize=7)    ]

    # plt.errorbar(spec_z, log_stellar_mass_Q50, yerr=[log_stellar_mass_lower_err, log_stellar_mass_upper_err], fmt='o', ecolor='black', alpha=0.5)
        
    for xi, yi, colour, style in zip(spec_z, log_stellar_mass_Q50, colours, marker_styles):
        
        plt.scatter(xi, yi, c=colour, marker=style, alpha=0.7, s=20)
    #--------------------------------------------------------------------

    plt.xlabel(r'Redshift $z_{spec}$', fontsize=18)
    plt.ylabel(r'Log Stellar Mass $log_{10}(M_*/M_\odot)$', fontsize=18)
    plt.grid(True)

    # plt.hexbin(spec_z, log_stellar_mass_Q50, cmap='Blues', alpha=1.0)

    ax.tick_params(
            axis="both",
            which="major",
            labelsize=14,
            length=8,
            width=1.5,
            direction="in",
            top=True,
            right=True
        )
    
    ax.xaxis.set_minor_locator(AutoMinorLocator(5))
    ax.yaxis.set_minor_locator(AutoMinorLocator(5))

    ax.tick_params(
        axis="both",
        which="minor",
        length=4,
        width=1,
        direction="in",
        top=True,
        right=True
    )

    add_marginal_hists(ax, x=spec_z, y=log_stellar_mass_Q50, bins=30, size="15%", pad=0.1, color="gray")

    redshift_bins(spec_z, log_stellar_mass_Q50, ax, "s", 8, "black")

    line_legend = ax.legend(loc='lower right', fontsize=15)
    ax.add_artist(line_legend)

    ax.legend(handles=legend_elements, title="Survey", loc='upper right', fontsize=13, title_fontsize=14, frameon=True, edgecolor='black', facecolor='white')

    fig_path = os.path.join(FIG_DIR, "stellar_mass_vs_redshift.png")
    fig.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

In [223]:
def plot_stellar_mass_vs_redshift_show_LRDs(sample_table):
    """
    Shows LRDs.
    """

    log_stellar_mass_Q16 = sample_table["stellar_mass_16"]
    log_stellar_mass_Q50 = sample_table["stellar_mass_50"]
    log_stellar_mass_Q84 = sample_table["stellar_mass_84"]

    log_stellar_mass_upper_err = log_stellar_mass_Q84 - log_stellar_mass_Q50
    log_stellar_mass_lower_err = log_stellar_mass_Q50 - log_stellar_mass_Q16

    spec_z = sample_table["input_redshift"]

    # -------------------------- Not colouring by survey, now showcasing LRDS -----------------------------
    fig, ax = plt.subplots(figsize=(10, 10))

    # --------------------------- Colouring and marker styles for each survey ----------------------------
    is_LRD = np.asarray(sample_table["is_LRD"].data, dtype=bool)
    
    colours = np.empty(len(is_LRD), dtype=object)
    marker_styles = np.empty(len(is_LRD), dtype='<U1')  # Use a string type for marker styles
    # z_order = np.empty(len(is_LRD), dtype='<U1')

    for i in range(len(is_LRD)):
        if is_LRD[i]:
            colours[i] = 'tab:red'
            marker_styles[i] = 'o'
            
        else:
            colours[i] = 'tab:blue'
            marker_styles[i] = 's'
            

    legend_elements = [
            Line2D([0], [0], marker='o', color='w',
                label='LRD',
                markerfacecolor='tab:red',
                markeredgecolor='0.7',
                linewidth=0.5,
                markersize=7),

            Line2D([0], [0], marker='s', color='w',
                label='Galaxy',
                markerfacecolor='tab:blue',
                markeredgecolor='0.3',
                linewidth=0.5,
                markersize=7),    
            ]

    # plt.errorbar(spec_z, log_stellar_mass_Q50, yerr=[log_stellar_mass_lower_err, log_stellar_mass_upper_err], fmt='o', ecolor='black', alpha=0.5)
        
    for xi, yi, colour, style in zip(spec_z, log_stellar_mass_Q50, colours, marker_styles):
        
        plt.scatter(xi, yi, c=colour, marker=style, alpha=0.7, s=20)
    #--------------------------------------------------------------------

    plt.xlabel(r'Redshift $z_{spec}$', fontsize=18)
    plt.ylabel(r'Log Stellar Mass $log_{10}(M_*/M_\odot)$', fontsize=18)
    plt.grid(True)

    # plt.hexbin(spec_z, log_stellar_mass_Q50, cmap='Blues', alpha=1.0)

    ax.tick_params(
            axis="both",
            which="major",
            labelsize=14,
            length=8,
            width=1.5,
            direction="in",
            top=True,
            right=True
        )
    
    ax.xaxis.set_minor_locator(AutoMinorLocator(5))
    ax.yaxis.set_minor_locator(AutoMinorLocator(5))

    ax.tick_params(
        axis="both",
        which="minor",
        length=4,
        width=1,
        direction="in",
        top=True,
        right=True
    )

    add_marginal_hists(ax, x=spec_z, y=log_stellar_mass_Q50, bins=30, size="15%", pad=0.1, color="gray")

    redshift_bins(spec_z, log_stellar_mass_Q50, ax, "s", 8, "black")

    line_legend = ax.legend(loc='lower right', fontsize=15)
    ax.add_artist(line_legend)

    ax.legend(handles=legend_elements, loc='upper right', fontsize=13, frameon=True, edgecolor='black', facecolor='white')

    fig_path = os.path.join(FIG_DIR, "stellar_mass_vs_redshift_LRDs.png")
    fig.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

In [224]:
def plot_stellar_mass_vs_M_UV(sample_table):

    z_bins = [[6.5, 7.5], [7.5, 8.5], [8.5, 9.5], [9.5, 10.5], [10.5, 11.5], [11.5, 13.5]]
    # Only going up to z=12.5 because there are very few galaxies at higher redshifts, and the bins would be empty.

    #Creating a figure of 8 subplots in 2x4 grid for each galaxy
    fig, axes = plt.subplots(2, 3, figsize=(20,10), constrained_layout=False)
    axes = axes.ravel()

    for i, z_bin in enumerate(z_bins):
        bin_mask = (sample_table["input_redshift"] >= z_bin[0]) & (sample_table["input_redshift"] < z_bin[1])
        bin_sample = sample_table[bin_mask]

        if len(bin_sample) > 5:
            print(f"Redshift bin {z_bin}: {len(bin_sample)} galaxies")

        ax = axes[i]

        log_stellar_mass_Q16 = bin_sample["stellar_mass_16"]
        log_stellar_mass_Q50 = bin_sample["stellar_mass_50"]
        log_stellar_mass_Q84 = bin_sample["stellar_mass_84"]

        log_stellar_mass_upper_err = log_stellar_mass_Q84 - log_stellar_mass_Q50
        log_stellar_mass_lower_err = log_stellar_mass_Q50 - log_stellar_mass_Q16

        M_uv_Q16 = bin_sample["M_UV_16"]
        M_uv_Q50 = bin_sample["M_UV_50"]
        M_uv_Q84 = bin_sample["M_UV_84"]

        # plt.errorbar(redshift, log_stellar_mass_Q50, yerr=[log_stellar_mass_lower_err, log_stellar_mass_upper_err], fmt='o', ecolor='black', alpha=0.5)
        # plt.scatter(M_uv_Q50, log_stellar_mass_Q50, color='steelblue', alpha=0.6)

        ax.set_xlabel(r'M$_{UV}$', fontsize=18)
        ax.set_ylabel(r'Log Stellar Mass $log_{10}(M_*/M_\odot)$', fontsize=18)
        ax.grid(True)

        ax.hexbin(M_uv_Q50, log_stellar_mass_Q50, cmap='Blues', alpha=1.0)

        ax.tick_params(
                axis="both",
                which="major",
                labelsize=14,
                length=8,
                width=1.5,
                direction="in",
                top=True,
                right=True
            )
    
        ax.xaxis.set_minor_locator(AutoMinorLocator(5))
        ax.yaxis.set_minor_locator(AutoMinorLocator(5))

        ax.tick_params(
            axis="both",
            which="minor",
            length=4,
            width=1,
            direction="in",
            top=True,
            right=True
        )

        add_marginal_hists(ax, x=M_uv_Q50, y=log_stellar_mass_Q50, bins=30, size="15%", pad=0.1, color="steelblue")

        if len(bin_sample) < 30:
            n_bins = 5
        else:
            n_bins = 8

        plot_median_bins(
            x_values=M_uv_Q50, y_values=log_stellar_mass_Q50, n_bins=n_bins, binned_axis_name="M$_{UV}$",
            ax=ax, marker_style="s", marker_size=8, colour="orange"
        )

        ax.legend(loc="lower left", fontsize=15)

        legend_text = f"Redshift bin: {z_bin[0]} - {z_bin[1]}"
        ax.text(0.05, 0.95, legend_text, transform=ax.transAxes, fontsize=15, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
    
    plt.tight_layout()
    fig_path = os.path.join(FIG_DIR, "stellar_mass_vs_M_UV.png")
    fig.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

In [225]:
if __name__ == "__main__":

    epochs_table_path = "/raid/scratch/work/alberttg/Masters_project/Data_inputs/EPOCHS-v2_EPOCHS_good_EAZY_sfhz_blue_agn_zfree_MASTER_Sel-F277W+F356W+F444W_v12_psfmatch_F444W_empirical+v13_psfmatch_F444W_empirical+v14_psfmatch_F444W_empirical.fits"

    # epochs_table_path = "/raid/scratch/work/austind/GALFIND_WORK/Catalogues/v12_psfmatch_F444W_empirical+v13_psfmatch_F444W_empirical+v14_psfmatch_F444W_empirical/ACS_WFC+NIRCam/EPOCHS-v2_EPOCHS_good_EAZY_sfhz_blue_agn_zfree/(0.32)as/EPOCHS-v2_EPOCHS_good_EAZY_sfhz_blue_agn_zfree_MASTER_Sel-F277W+F356W+F444W_v12_psfmatch_F444W_empirical+v13_psfmatch_F444W_empirical+v14_psfmatch_F444W_empirical.fits""

    sex_tab, bagpipes_zfix, bagpipes_zgauss, selection_tab = open_catalogue_data(epochs_table_path)

    sample = masked_table(sex_tab, bagpipes_zfix, bagpipes_zgauss, selection_tab)

    plot_stellar_mass_vs_redshift(sample)

    plot_stellar_mass_vs_redshift_show_LRDs(sample)

    plot_stellar_mass_vs_M_UV(sample)

    

Check inital length:  2517
Check length after filtering out brown dwarfs:  2425
Redshift bin [6.5, 7.5]: 1335 galaxies
Redshift bin [7.5, 8.5]: 555 galaxies
Redshift bin [8.5, 9.5]: 226 galaxies
Redshift bin [9.5, 10.5]: 125 galaxies
Redshift bin [10.5, 11.5]: 103 galaxies
Redshift bin [11.5, 13.5]: 81 galaxies
